# 🧬 个人人格 LoRA · Kaggle 一键训练

**用途**：用你在本地产出并审核过的 `train.jsonl` / `validation.jsonl`，在 Kaggle 免费 T4 上训练专属人格 LoRA 适配器。

## 使用步骤
1. **上传数据集**：Kaggle 网页 → **Datasets → New Dataset** → 上传 `train.jsonl` + `validation.jsonl`（保持 **Private**）
2. **新建 Notebook**：File → Import Notebook → 上传本文件
3. **挂载数据**：右侧 **Add Input → Your Datasets** → 选中你刚建的数据集
4. **开启算力**：Settings → Accelerator → **GPU T4 x2**（选 1 张 T4 即可）→ Internet → **On**
5. **Run All** 🚀（默认 Qwen 6B + 4bit + LoRA r16，约 2-3 小时）

## 产出
- `persona_lora_adapter.zip`：几十 MB 的 LoRA 适配器（必产出）
- 可选：合并 16bit 完整模型 / Q4_K_M GGUF（Ollama 直接可用）

> 预置超参已按「风格复刻」调优：LoRA r=16 / epoch 3 / lr 2e-4 / 早停防死记硬背，**无需调参**。

In [ ]:
# ============ 配置区（一般只需看这里） ============
BASE_MODEL = "Qwen/Qwen3.5-6B-Instruct"
# 备选梯队（改这一行即可）：
#   轻量版: "Qwen/Qwen2.5-3B-Instruct"   —— 极低配置可跑，上限低
#   进阶版: "Qwen/Qwen3.5-14B-Instruct"  —— 效果更强，T4 需谨慎（建议 3B/6B）

LORA_R = 16           # 8 或 16；风格复刻推荐 16
EPOCHS = 3            # 2~4；配合早停，杜绝过拟合死记硬背
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 2048
BATCH_SIZE = 2
GRAD_ACCUM = 4        # 有效批次 = 8

print(f"基座: {BASE_MODEL} | LoRA r={LORA_R} | epochs={EPOCHS}")

## 1. 安装 Unsloth（极速训练 + 低显存 4bit）

In [ ]:
%%capture
!pip install unsloth

In [ ]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,            # T4 自动 fp16 / 新卡自动 bf16
    load_in_4bit=True,     # QLoRA 4bit 量化训练
)
print("模型加载完成 ✅")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_R,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print(f"LoRA 挂载完成：可训练参数占比仅 {model.print_trainable_parameters() or ''}")

## 2. 加载你的个人数据集（右侧 Add Input 挂载后自动发现）

In [ ]:
import glob
from datasets import load_dataset

train_files = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
val_files = glob.glob("/kaggle/input/**/validation.jsonl", recursive=True)
assert train_files, "未找到 train.jsonl：请在右侧 Add Input 添加你的数据集后再运行"

data_files = {"train": train_files[0]}
if val_files:
    data_files["validation"] = val_files[0]
ds = load_dataset("json", data_files=data_files)
print(ds)
print("\n样本示例:")
ex = ds["train"][0]["messages"]
for m in ex[:4]:
    print(f'  [{m["role"]}] {m["content"][:60]}')

In [ ]:
def formatting_func(example):
    """messages → ChatML 文本（Qwen 官方对话模板）。"""
    return tokenizer.apply_chat_template(example["messages"], tokenize=False)

print(formatting_func(ds["train"][0])[:500])

## 3. 训练（预置最优超参 + 验证集早停）

In [ ]:
from transformers import EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer

has_val = "validation" in ds
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"] if has_val else None,
    formatting_func=formatting_func,
    args=SFTConfig(
        output_dir="outputs",
        max_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        logging_steps=10,
        eval_strategy="steps" if has_val else "no",
        eval_steps=25,
        save_strategy="steps" if has_val else "no",
        save_steps=25,
        load_best_model_at_end=has_val,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        weight_decay=0.01,
        seed=42,
        report_to="none",
    ),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] if has_val else None,
)
print("开始训练 🚀（验证 loss 连续 3 次不下降自动早停）")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_loss = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_loss = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure(figsize=(10, 5))
plt.plot(*zip(*train_loss), label="train_loss", color="#1f77b4")
if eval_loss:
    plt.plot(*zip(*eval_loss), label="eval_loss", color="#d62728", marker="o")
    plt.axvline(eval_loss[-1][0], ls="--", c="gray", alpha=0.5)
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("Loss Curves")
plt.show()
print(f"最终 train loss: {train_loss[-1][1]:.4f}  |  eval loss: {eval_loss[-1][1]:.4f}" if eval_loss else f"最终 train loss: {train_loss[-1][1]:.4f}")

## 4. 保存 LoRA 适配器（核心产物：几十 MB）

In [ ]:
model.save_pretrained("persona_lora_adapter")
tokenizer.save_pretrained("persona_lora_adapter")
!zip -r -q persona_lora_adapter.zip persona_lora_adapter
import os
size_mb = os.path.getsize("persona_lora_adapter.zip") / 1e6
print(f"persona_lora_adapter.zip: {size_mb:.1f} MB")

from IPython.display import FileLink
FileLink("persona_lora_adapter.zip")

## 5. 快速试玩（看看它学会了你的口吻没）

In [ ]:
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": "你是「我」本人的数字分身，请完全用你自己的口吻、语气词、标点习惯和句式风格自然回复。"},
    {"role": "user", "content": "这周末有空吗？出来吃个饭呗，好久没见了"},
]
inputs = tokenizer.apply_chat_template(
    test_messages, return_tensors="pt", add_generation_prompt=True
).to("cuda")
outputs = model.generate(
    input_ids=inputs, max_new_tokens=150,
    temperature=0.8, top_p=0.9, repetition_penalty=1.1,
)
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))

## 6.（可选）导出合并模型 / GGUF（Ollama 直接加载）

In [ ]:
# 想本地 Ollama 直接跑 → 改 EXPORT_GGUF = True 再运行本格（多花几分钟）
# 想用 Mac MLX → EXPORT_MERGED = True，下载 persona_merged 后按 docs/hardware.md 转 MLX

EXPORT_GGUF = False
EXPORT_MERGED = False

if EXPORT_MERGED:
    model.save_pretrained_merged("persona_merged", tokenizer, save_method="merged_16bit")
    !zip -r -q persona_merged.zip persona_merged
    display(FileLink("persona_merged.zip"))

if EXPORT_GGUF:
    model.save_pretrained_gguf("persona_gguf", tokenizer, quantization_method="q4_k_m")
    !zip -r -q persona_gguf_q4km.zip persona_gguf
    display(FileLink("persona_gguf_q4km.zip"))

## ✅ 训练完成，接下来？

1. 下载 `persona_lora_adapter.zip`（或 GGUF zip）到本地
2. 本地部署：
   - **Ollama**：`python scripts/make_ollama.py --gguf persona_gguf-...gguf --name my-persona`
   - **Mac MLX**：见 `docs/hardware.md`（M1 8G 也能流畅跑）
   - **对话测试**：`python scripts/chat_test.py`
3. 效果不满意？回到本地工作台继续攒样本（对话采集 / IM 抽取），重新导出训练即可